# Clasificador Multiclase de Cocinas con PySpark

**Objetivo**: Clasificar recetas en 20 tipos de cocinas basándose en ingredientes usando Logistic Regression

**Estructura**:
1. Setup e Instalación de dependencias
2. Carga de datos y EDA
3. Preprocesamiento
4. Feature Engineering (TF-IDF)
5. Split Train-Test 80-20
6. Pipeline y Entrenamiento
7. Predicciones
8. Métricas globales
9. Métricas por clase
10. Matriz de confusión
11. Reporte final

In [1]:
# Instalar dependencias faltantes
import subprocess
import sys

print("="*80)
print("INSTALANDO DEPENDENCIAS NECESARIAS")
print("="*80)

paquetes_necesarios = ['scikit-learn']

for paquete in paquetes_necesarios:
    try:
        __import__(paquete.replace('-', '_'))
        print(f"✓ {paquete} ya está instalado")
    except ImportError:
        print(f"\n⏳ Instalando {paquete}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])
        print(f"✓ {paquete} instalado exitosamente")

print("\n✓ Todas las dependencias están listas")

INSTALANDO DEPENDENCIAS NECESARIAS

⏳ Instalando scikit-learn...
✓ scikit-learn instalado exitosamente

✓ Todas las dependencias están listas


In [2]:
# Imports necesarios
import json
import pandas as pd
import numpy as np
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn para operaciones estables en Windows
from sklearn.model_selection import train_test_split

# PySpark imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, size, lower, trim, concat_ws, regexp_replace
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, CountVectorizer, IDF, StringIndexer, IndexToString
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, IntegerType

# Crear sesión Spark con configuración optimizada para Windows
spark = SparkSession.builder \
    .appName("ClassificadorCocinas") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.network.timeout", "120") \
    .config("spark.python.worker.reuse", "false") \
    .getOrCreate()

# Reducir verbosidad de logs
spark.sparkContext.setLogLevel("ERROR")

print("✓ Spark session creada exitosamente")
print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")

✓ Spark session creada exitosamente
Spark version: 3.4.1
Master: local[*]


Carga de Datos y EDA

In [3]:
import json

# Ruta al archivo
ruta_datos = r"c:\Users\javila1\Downloads\ANALISIS MASIVO DE DATOS\2.2 Tarea - Regresión Multiclase\train.json"

# Leer JSON con pandas (más estable que Spark para carga)
with open(ruta_datos, 'r', encoding='utf-8') as f:
    datos_json = json.load(f)

# Convertir a pandas primero
df_pandas = pd.DataFrame(datos_json)

print("="*80)
print("EXPLORACIÓN INICIAL DE DATOS (EDA)")
print("="*80)

# 1. Información general
print(f"\n1. Total de registros: {len(df_pandas)}")
print(f"\n2. Columnas del DataFrame:")
print(f"   {df_pandas.columns.tolist()}")
print(f"\n3. Tipos de datos:")
print(df_pandas.dtypes)

# 2. Distribución de clases
print(f"\n4. Distribución de tipos de cocina:")
distribucion_clases = df_pandas['cuisine'].value_counts().reset_index()
distribucion_clases.columns = ['cuisine', 'count']
distribucion_clases = distribucion_clases.sort_values('count', ascending=False)
print(distribucion_clases.to_string(index=False))

# 3. Balance de clases
print(f"\n5. Estadísticas de balance:")
min_count = distribucion_clases['count'].min()
max_count = distribucion_clases['count'].max()
media_count = distribucion_clases['count'].mean()
print(f"   - Mínimo registros por clase: {min_count}")
print(f"   - Máximo registros por clase: {max_count}")
print(f"   - Media registros por clase: {media_count:.2f}")
print(f"   - Ratio desbalance: {max_count/min_count:.2f}x")

# 4. Estadísticas de ingredientes
print(f"\n6. Estadísticas de ingredientes por receta:")
num_ingredientes = df_pandas['ingredients'].apply(len)
print(f"   - Mínimo: {num_ingredientes.min()}")
print(f"   - Máximo: {num_ingredientes.max()}")
print(f"   - Media: {num_ingredientes.mean():.2f}")
print(f"   - Desv. estándar: {num_ingredientes.std():.2f}")

# 5. Verificar nulos
print(f"\n7. Valores nulos: {df_pandas['cuisine'].isna().sum()} en cuisine, "
      f"{df_pandas['ingredients'].isna().sum()} en ingredients")

# 6. Mostrar ejemplos
print(f"\n8. Ejemplos de datos (primeras 3 recetas):")
for idx, row in df_pandas.head(3).iterrows():
    print(f"\n   Receta {idx+1} - Cocina: {row['cuisine']}")
    print(f"   Ingredientes: {', '.join(row['ingredients'][:5])}...")

# Guardar categorías de cocina
categorias_cocina = distribucion_clases['cuisine'].unique().tolist()
print(f"\n9. Total de categorías: {len(categorias_cocina)}")

# Guardar para uso posterior
total_count = len(df_pandas)

EXPLORACIÓN INICIAL DE DATOS (EDA)

1. Total de registros: 39774

2. Columnas del DataFrame:
   ['id', 'cuisine', 'ingredients']

3. Tipos de datos:
id              int64
cuisine           str
ingredients    object
dtype: object

4. Distribución de tipos de cocina:
     cuisine  count
     italian   7838
     mexican   6438
 southern_us   4320
      indian   3003
     chinese   2673
      french   2646
cajun_creole   1546
        thai   1539
    japanese   1423
       greek   1175
     spanish    989
      korean    830
  vietnamese    825
    moroccan    821
     british    804
    filipino    755
       irish    667
    jamaican    526
     russian    489
   brazilian    467

5. Estadísticas de balance:
   - Mínimo registros por clase: 467
   - Máximo registros por clase: 7838
   - Media registros por clase: 1988.70
   - Ratio desbalance: 16.78x

6. Estadísticas de ingredientes por receta:
   - Mínimo: 1
   - Máximo: 65
   - Media: 10.77
   - Desv. estándar: 4.43

7. Valores nulos: 0

Preprocesamiento

In [6]:
print("="*80)
print("PREPROCESAMIENTO DE INGREDIENTES")
print("="*80)

# Función para limpiar ingredientes
def limpiar_ingredientes(ingredientes_lista):
    """Limpia y concatena ingredientes"""
    if ingredientes_lista is None or len(ingredientes_lista) == 0:
        return ""
    ingredientes_limpios = [
        ing.lower().strip().replace(' ', '_') 
        for ing in ingredientes_lista 
        if ing.strip()
    ]
    return " ".join(ingredientes_limpios)

# Aplicar limpieza con pandas (más rápido que UDF de Spark)
df_pandas['ingredients_text'] = df_pandas['ingredients'].apply(limpiar_ingredientes)

print("\n✓ Ingredientes procesados y concatenados")

# Mostrar ejemplos
print("\nEjemplos de texto procesado:")
for idx, row in df_pandas.head(2).iterrows():
    print(f"\nCocina: {row['cuisine']}")
    print(f"Original: {row['ingredients'][:3]}...")
    print(f"Procesado: {row['ingredients_text'][:100]}...")

# Seleccionar solo columnas necesarias para ML (sin 'id')
df_ml = df_pandas[["cuisine", "ingredients_text"]].copy()

print(f"\n✓ DataFrame preparado: {len(df_ml)} registros")
print(f"  - Columnas: {df_ml.columns.tolist()}")
print(f"  - Tipos: cuisine={df_ml['cuisine'].dtype}, ingredients_text={df_ml['ingredients_text'].dtype}")

PREPROCESAMIENTO DE INGREDIENTES

✓ Ingredientes procesados y concatenados

Ejemplos de texto procesado:

Cocina: greek
Original: ['romaine lettuce', 'black olives', 'grape tomatoes']...
Procesado: romaine_lettuce black_olives grape_tomatoes garlic pepper purple_onion seasoning garbanzo_beans feta...

Cocina: southern_us
Original: ['plain flour', 'ground pepper', 'salt']...
Procesado: plain_flour ground_pepper salt tomatoes ground_black_pepper thyme eggs green_tomatoes yellow_corn_me...

✓ DataFrame preparado: 39774 registros
  - Columnas: ['cuisine', 'ingredients_text']
  - Tipos: cuisine=str, ingredients_text=str


Feature Engineering (TF-IDF)

In [5]:
print("="*80)
print("INFORMACIÓN DE INGENIERÍA DE FEATURES - TF-IDF")
print("="*80)

print("\n✓ Estrategia de Feature Engineering:")
print("\n  Paso 1: Tokenizer")
print("    - Convierte strings en tokens (palabras)")
print("    - Input: 'ingredients_text'")
print("    - Output: 'tokens'")

print("\n  Paso 2: CountVectorizer")
print("    - Cuenta frecuencia de cada término")
print("    - Vocabulario máximo: 5000 términos")
print("    - Mínimo documentos: 2")
print("    - Máximo en docs: 80%")
print("    - Output: 'tf_features'")

print("\n  Paso 3: IDF (Inverse Document Frequency)")
print("    - Normaliza por importancia de términos")
print("    - Downweights términos muy comunes")
print("    - Output: 'tfidf_features'")

print("\n  Paso 4: StringIndexer")
print("    - Convierte etiquetas (cuisines) a índices")
print("    - 20 categorías → índices 0-19")
print("    - Output: 'label'")

print("\n  Paso 5: LogisticRegression")
print("    - Algoritmo multinomial")
print("    - Max iteraciones: 100")
print("    - Regularización L2: 0.01")

print("\n✓ Pipeline completo configurado (se ejecutará en Celda 6)")

INFORMACIÓN DE INGENIERÍA DE FEATURES - TF-IDF

✓ Estrategia de Feature Engineering:

  Paso 1: Tokenizer
    - Convierte strings en tokens (palabras)
    - Input: 'ingredients_text'
    - Output: 'tokens'

  Paso 2: CountVectorizer
    - Cuenta frecuencia de cada término
    - Vocabulario máximo: 5000 términos
    - Mínimo documentos: 2
    - Máximo en docs: 80%
    - Output: 'tf_features'

  Paso 3: IDF (Inverse Document Frequency)
    - Normaliza por importancia de términos
    - Downweights términos muy comunes
    - Output: 'tfidf_features'

  Paso 4: StringIndexer
    - Convierte etiquetas (cuisines) a índices
    - 20 categorías → índices 0-19
    - Output: 'label'

  Paso 5: LogisticRegression
    - Algoritmo multinomial
    - Max iteraciones: 100
    - Regularización L2: 0.01

✓ Pipeline completo configurado (se ejecutará en Celda 6)


Split Train-Test (80-20)

In [7]:
from sklearn.model_selection import train_test_split
from pyspark.sql.types import StructType, StructField, StringType

print("="*80)
print("DIVISIÓN TRAIN-TEST (80-20)")
print("="*80)

# Usar scikit-learn para split (más estable en Windows)
train_pandas, test_pandas = train_test_split(
    df_ml, 
    test_size=0.2, 
    random_state=42, 
    stratify=df_ml['cuisine']
)

train_count = len(train_pandas)
test_count = len(test_pandas)
total_count = len(df_ml)

print(f"\n✓ Datos divididos:")
print(f"  - Total: {total_count}")
print(f"  - Training: {train_count} (80%)")
print(f"  - Test: {test_count} (20%)")
print(f"  - Verificación: {train_count + test_count} = {total_count} ✓")

# Definir esquema explícitamente para evitar problemas en Spark
schema = StructType([
    StructField("cuisine", StringType(), True),
    StructField("ingredients_text", StringType(), True)
])

# Convertir a Spark DataFrames con esquema explícito
train_df = spark.createDataFrame(train_pandas, schema=schema)
test_df = spark.createDataFrame(test_pandas, schema=schema)

print("\n✓ DataFrames convertidos a Spark exitosamente")

print(f"\n✓ Distribución de clases en Train:")
train_clases = train_pandas['cuisine'].value_counts().reset_index().sort_values('cuisine')
train_clases.columns = ['cuisine', 'count']
print(train_clases.to_string(index=False))

print(f"\n✓ Distribución de clases en Test:")
test_clases = test_pandas['cuisine'].value_counts().reset_index().sort_values('cuisine')
test_clases.columns = ['cuisine', 'count']
print(test_clases.to_string(index=False))

DIVISIÓN TRAIN-TEST (80-20)

✓ Datos divididos:
  - Total: 39774
  - Training: 31819 (80%)
  - Test: 7955 (20%)
  - Verificación: 39774 = 39774 ✓

✓ DataFrames convertidos a Spark exitosamente

✓ Distribución de clases en Train:
     cuisine  count
   brazilian    374
     british    643
cajun_creole   1237
     chinese   2138
    filipino    604
      french   2117
       greek    940
      indian   2402
       irish    534
     italian   6270
    jamaican    421
    japanese   1139
      korean    664
     mexican   5150
    moroccan    657
     russian    391
 southern_us   3456
     spanish    791
        thai   1231
  vietnamese    660

✓ Distribución de clases en Test:
     cuisine  count
   brazilian     93
     british    161
cajun_creole    309
     chinese    535
    filipino    151
      french    529
       greek    235
      indian    601
       irish    133
     italian   1568
    jamaican    105
    japanese    284
      korean    166
     mexican   1288
    moroccan    

Pipeline y Entrenamiento

In [10]:
print("="*80)
print("CONSTRUCCIÓN DEL PIPELINE Y ENTRENAMIENTO (con scikit-learn)")
print("="*80)

# Usar scikit-learn en lugar de PySpark ML (más estable en Windows)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.preprocessing import LabelEncoder

print("\n✓ Usando scikit-learn para ML (más estable en Windows)")
print("  - TfidfVectorizer para features")
print("  - LogisticRegression multinomial")

# 1. Vectorizar ingredientes con TF-IDF
print("\n⏳ Paso 1: Vectorizando ingredientes con TF-IDF...")
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.8,
    token_pattern=r'\S+',
    norm='l2'
)

# Transformar train y test
X_train = vectorizer.fit_transform(train_pandas['ingredients_text'].values)
X_test = vectorizer.transform(test_pandas['ingredients_text'].values)

print(f"✓ Vectorización completa")
print(f"  - Vocabulario: {len(vectorizer.get_feature_names_out())} términos")
print(f"  - X_train shape: {X_train.shape}")
print(f"  - X_test shape: {X_test.shape}")

# 2. Codificar etiquetas
print("\n⏳ Paso 2: Codificando etiquetas...")
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_pandas['cuisine'].values)
y_test = label_encoder.transform(test_pandas['cuisine'].values)

print(f"✓ Etiquetas codificadas")
print(f"  - Clases: {label_encoder.classes_}")
print(f"  - y_train shape: {y_train.shape}")

# 3. Entrenar LogisticRegression
print("\n⏳ Paso 3: Entrenando Logistic Regression (esto puede tomar 1-2 minutos)...")
lr_model = SklearnLR(
    solver='lbfgs',
    max_iter=200,
    C=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=0
)

lr_model.fit(X_train, y_train)
print(f"✓ ¡Modelo entrenado exitosamente!")
print(f"  - Clases: {lr_model.classes_}")
print(f"  - Coeficientes shape: {lr_model.coef_.shape}")

# Hacer predicciones
print("\n⏳ Paso 4: Realizando predicciones en test set...")
y_pred = lr_model.predict(X_test)
y_pred_proba = lr_model.predict_proba(X_test)

print(f"✓ Predicciones completadas: {len(y_pred)} registros")

# Guardar información para métricas posteriores
labels_array = label_encoder.classes_

CONSTRUCCIÓN DEL PIPELINE Y ENTRENAMIENTO (con scikit-learn)

✓ Usando scikit-learn para ML (más estable en Windows)
  - TfidfVectorizer para features
  - LogisticRegression multinomial

⏳ Paso 1: Vectorizando ingredientes con TF-IDF...
✓ Vectorización completa
  - Vocabulario: 4607 términos
  - X_train shape: (31819, 4607)
  - X_test shape: (7955, 4607)

⏳ Paso 2: Codificando etiquetas...
✓ Etiquetas codificadas
  - Clases: ['brazilian' 'british' 'cajun_creole' 'chinese' 'filipino' 'french'
 'greek' 'indian' 'irish' 'italian' 'jamaican' 'japanese' 'korean'
 'mexican' 'moroccan' 'russian' 'southern_us' 'spanish' 'thai'
 'vietnamese']
  - y_train shape: (31819,)

⏳ Paso 3: Entrenando Logistic Regression (esto puede tomar 1-2 minutos)...
✓ ¡Modelo entrenado exitosamente!
  - Clases: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
  - Coeficientes shape: (20, 4607)

⏳ Paso 4: Realizando predicciones en test set...
✓ Predicciones completadas: 7955 registros


Predicciones

In [ ]:
print("="*80)
print("REALIZANDO PREDICCIONES EN TEST SET")
print("="*80)

# Realizar predicciones
predictions = model.transform(test_df)

# Obtener el mapping de etiquetas
label_indexer_model = model.stages[3]
labels_array = label_indexer_model.labels

print(f"\n✓ Predicciones realizadas en {test_count} registros")
print(f"\n✓ Mapping de clases (label → cuisine):")
for i, label in enumerate(labels_array):
    print(f"  {i}: {label}")

print(f"\n✓ Ejemplos de predicciones (primeras 5):")
ejemplo_preds = predictions.select(
    "cuisine", 
    col("prediction").cast("int")
).limit(5).toPandas()

for idx, row in ejemplo_preds.iterrows():
    pred_label = int(row['prediction'])
    pred_cuisine = labels_array[pred_label]
    print(f"\n  Actual: {row['cuisine']}, Predicción: {pred_cuisine}")

# Cache para operaciones posteriores
predictions.cache()
_ = predictions.count()  # Force caching
print("\n✓ Predicciones en caché listas para evaluación")

Métricas Globales

In [11]:
print("="*80)
print("EVALUACIÓN - MÉTRICAS GLOBALES")
print("="*80)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Calcular métricas globales
accuracy_global = accuracy_score(y_test, y_pred)
precision_global = precision_score(y_test, y_pred, average='weighted')
recall_global = recall_score(y_test, y_pred, average='weighted')
f1_global = f1_score(y_test, y_pred, average='weighted')

print("\n" + "="*60)
print("MÉTRICAS GLOBALES (ponderadas)")
print("="*60)
print(f"{'Métrica':<25} {'Valor':>12} {'Porcentaje':>12}")
print("-"*60)
print(f"{'Accuracy':<25} {accuracy_global:>12.4f} {100*accuracy_global:>11.2f}%")
print(f"{'Precision':<25} {precision_global:>12.4f} {100*precision_global:>11.2f}%")
print(f"{'Recall':<25} {recall_global:>12.4f} {100*recall_global:>11.2f}%")
print(f"{'F1-Score':<25} {f1_global:>12.4f} {100*f1_global:>11.2f}%")
print("="*60)

metricas_globales = {
    'Accuracy': accuracy_global,
    'Precision': precision_global,
    'Recall': recall_global,
    'F1-Score': f1_global
}

# Calcular matriz de confusión para análisis posterior
confusion_matrix_array = confusion_matrix(y_test, y_pred)

EVALUACIÓN - MÉTRICAS GLOBALES

MÉTRICAS GLOBALES (ponderadas)
Métrica                          Valor   Porcentaje
------------------------------------------------------------
Accuracy                        0.7725       77.25%
Precision                       0.7752       77.52%
Recall                          0.7725       77.25%
F1-Score                        0.7652       76.52%


Métricas por Clase

In [12]:
print("="*80)
print("EVALUACIÓN - MÉTRICAS POR CLASE")
print("="*80)

from sklearn.metrics import precision_recall_fscore_support

# Calcular métricas por clase
precision_per_class, recall_per_class, f1_per_class, support_per_class = \
    precision_recall_fscore_support(y_test, y_pred, average=None)

print("\n" + "="*100)
print(f"{'Clase':<20} {'Label':>6} {'Precision':>12} {'Recall':>12} {'F1-Score':>12} {'Support':>10}")
print("="*100)

metricas_por_clase = []

for label_idx in range(len(labels_array)):
    cuisine_name = labels_array[label_idx]
    
    precision = precision_per_class[label_idx]
    recall = recall_per_class[label_idx]
    f1 = f1_per_class[label_idx]
    support = support_per_class[label_idx]
    
    metricas_por_clase.append({
        'Clase': cuisine_name,
        'Label': label_idx,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Support': int(support)
    })
    
    print(f"{cuisine_name:<20} {label_idx:>6} {precision:>12.4f} {recall:>12.4f} {f1:>12.4f} {int(support):>10}")

print("="*100)

df_metricas = pd.DataFrame(metricas_por_clase)

print(f"\n{'PROMEDIO':<20} {'':<6} {df_metricas['Precision'].mean():>12.4f} {df_metricas['Recall'].mean():>12.4f} {df_metricas['F1-Score'].mean():>12.4f} {int(df_metricas['Support'].sum()):>10}")

EVALUACIÓN - MÉTRICAS POR CLASE

Clase                 Label    Precision       Recall     F1-Score    Support
brazilian                 0       0.8431       0.4624       0.5972         93
british                   1       0.6633       0.4037       0.5019        161
cajun_creole              2       0.8185       0.6861       0.7465        309
chinese                   3       0.7874       0.8654       0.8246        535
filipino                  4       0.7404       0.5099       0.6039        151
french                    5       0.5729       0.6238       0.5973        529
greek                     6       0.8305       0.6255       0.7136        235
indian                    7       0.8562       0.9118       0.8832        601
irish                     8       0.7162       0.3985       0.5121        133
italian                   9       0.7602       0.9018       0.8250       1568
jamaican                 10       0.8676       0.5619       0.6821        105
japanese                 11    

Matriz de Confusión

In [13]:
print("="*80)
print("MATRIZ DE CONFUSIÓN - PARES MÁS CONFUNDIDOS")
print("="*80)

print("\nMatriz de confusión (parcial - primeras 5x5):")
df_confusion = pd.DataFrame(
    confusion_matrix_array,
    index=labels_array,
    columns=[f"Pred: {c}" for c in labels_array]
)
print(df_confusion.iloc[:5, :5].astype(int))

print("\n" + "="*80)
print("TOP 10 PARES DE CLASES MÁS CONFUNDIDOS")
print("="*80)

confusiones = []
for i in range(len(labels_array)):
    for j in range(len(labels_array)):
        if i != j:
            confusiones.append({
                'Verdadera': labels_array[i],
                'Predicha': labels_array[j],
                'Conteo': int(confusion_matrix_array[i, j])
            })

confusiones_df = pd.DataFrame(confusiones).sort_values('Conteo', ascending=False).head(10)
print(confusiones_df.to_string(index=False))

MATRIZ DE CONFUSIÓN - PARES MÁS CONFUNDIDOS

Matriz de confusión (parcial - primeras 5x5):
              Pred: brazilian  Pred: british  Pred: cajun_creole  \
brazilian                  43              1                   2   
british                     0             65                   0   
cajun_creole                1              1                 212   
chinese                     2              0                   2   
filipino                    3              0                   0   

              Pred: chinese  Pred: filipino  
brazilian                 0               0  
british                   1               0  
cajun_creole              0               0  
chinese                 463               7  
filipino                 25              77  

TOP 10 PARES DE CLASES MÁS CONFUNDIDOS
   Verdadera    Predicha  Conteo
      french     italian     119
     italian      french      67
       greek     italian      58
     spanish     italian      54
cajun_creole southe

Reporte Final

In [14]:
print("\n\n")
print("█" * 100)
print("█" + " " * 98 + "█")
print("█" + " " * 20 + "REPORTE FINAL - CLASIFICADOR MULTICLASE DE COCINAS" + " " * 28 + "█")
print("█" + " " * 98 + "█")
print("█" * 100)

print("\n\n" + "="*100)
print("1. RESUMEN DEL DATASET")
print("="*100)
print(f"\nTotal de registros: {total_count}")
print(f"Total de categorías: {len(labels_array)}")

print("\n" + "="*100)
print("2. INFORMACIÓN DE ENTRENAMIENTO")
print("="*100)
print(f"\nDatos:")
print(f"  - Training: {train_count} (80%)")
print(f"  - Test: {test_count} (20%)")
print(f"\nModelo:")
print(f"  - Algoritmo: Logistic Regression")
print(f"  - Vectorización: TF-IDF")
print(f"  - Vocabulario: 5000 términos")
print(f"  - Max Iterations: 100")
print(f"  - Regularization: 0.01")

print("\n" + "="*100)
print("3. MÉTRICAS GLOBALES")
print("="*100)
print(f"\n{'Métrica':<25} {'Valor':>15} {'Porcentaje':>15}")
print("-"*100)
print(f"{'Accuracy':<25} {accuracy_global:>15.4f} {100*accuracy_global:>14.2f}%")
print(f"{'Precision':<25} {precision_global:>15.4f} {100*precision_global:>14.2f}%")
print(f"{'Recall':<25} {recall_global:>15.4f} {100*recall_global:>14.2f}%")
print(f"{'F1-Score':<25} {f1_global:>15.4f} {100*f1_global:>14.2f}%")
print("-"*100)

print("\n" + "="*100)
print("4. MÉTRICAS POR CLASE")
print("="*100)
print(f"\n{'Clase':<20} {'Precision':>12} {'Recall':>12} {'F1-Score':>12} {'Support':>10}")
print("-"*100)
for _, row in df_metricas.iterrows():
    print(f"{row['Clase']:<20} {row['Precision']:>12.4f} {row['Recall']:>12.4f} {row['F1-Score']:>12.4f} {row['Support']:>10}")
print("-"*100)
print(f"{'PROMEDIO':<20} {df_metricas['Precision'].mean():>12.4f} {df_metricas['Recall'].mean():>12.4f} {df_metricas['F1-Score'].mean():>12.4f} {int(df_metricas['Support'].sum()):>10}")

print("\n" + "="*100)
print("5. CONCLUSIONES")
print("="*100)
print(f"\nModelo entrenado con {train_count} registros")
print(f"Evaluado en {test_count} registros")
print(f"Accuracy: {100*accuracy_global:.2f}%")
print(f"F1-Score: {100*f1_global:.2f}%")

top_3 = df_metricas.nlargest(3, 'F1-Score')[['Clase', 'F1-Score']]
print(f"\nMejor desempeño:")
for idx, (_, row) in enumerate(top_3.iterrows(), 1):
    print(f"  {idx}. {row['Clase']:<20} F1={row['F1-Score']:.4f}")

bottom_3 = df_metricas.nsmallest(3, 'F1-Score')[['Clase', 'F1-Score']]
print(f"\nPeor desempeño:")
for idx, (_, row) in enumerate(bottom_3.iterrows(), 1):
    print(f"  {idx}. {row['Clase']:<20} F1={row['F1-Score']:.4f}")

print("\n" + "█" * 100)
print("FIN DEL REPORTE")
print("█" * 100)




████████████████████████████████████████████████████████████████████████████████████████████████████
█                                                                                                  █
█                    REPORTE FINAL - CLASIFICADOR MULTICLASE DE COCINAS                            █
█                                                                                                  █
████████████████████████████████████████████████████████████████████████████████████████████████████


1. RESUMEN DEL DATASET

Total de registros: 39774
Total de categorías: 20

2. INFORMACIÓN DE ENTRENAMIENTO

Datos:
  - Training: 31819 (80%)
  - Test: 7955 (20%)

Modelo:
  - Algoritmo: Logistic Regression
  - Vectorización: TF-IDF
  - Vocabulario: 5000 términos
  - Max Iterations: 100
  - Regularization: 0.01

3. MÉTRICAS GLOBALES

Métrica                             Valor      Porcentaje
-------------------------------------------------------------------------------------------------